<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG-Chain mit LangChain
</b></font> </br></p>

---

In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M13-RAG-Chain"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 35.237.83.154
Hostname: 154.83.237.35.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/New_

In [2]:
#@title 📦 Pakete installieren{ display-mode: "form" }
install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

🔄 Installiere markitdown[all]...
✅ markitdown[all] erfolgreich installiert und importiert
🔄 Installiere langchain_huggingface...
✅ langchain_huggingface erfolgreich installiert und importiert
🔄 Installiere unstructured[all-docs]>=0.11.2...
✅ unstructured[all-docs]>=0.11.2 erfolgreich installiert und importiert


# 1 | Übersicht
---

In *RAG-Konzepte & Embeddings* haben wir Embeddings verstanden, in *ChromaDB & Indexing* ChromaDB kennengelernt.  
Jetzt **verbinden** wir alles: eine vollständige **RAG-Chain** mit LangChain.

## RAG-Chain vs. RAG-Agent

| Eigenschaft | RAG-Chain | RAG-Agent |
|-------------|-----------|----------|
| Ablauf | Deterministisch (immer gleich) | Dynamisch (entscheidet selbst) |
| Tool-Nutzung | Nein | Ja, inklusive RAG als Tool |
| Kontrolle | Vollständig | Flexibel, aber weniger vorhersehbar |
| Einstieg | ✅ Einfacher | Komplexer (M11) |

## Was wir bauen

Eine RAG-Chain, die:
1. Dokumente in ChromaDB indexiert (Phase 1)
2. Bei einer Anfrage relevante Chunks abruft (Retrieval)
3. Den LLM mit dem Kontext aufruft und antwortet (Generation)

In [3]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart LR
    subgraph Setup["Setup (einmalig)"]
        DOCS["📄 Dokumente"] --> SPLIT["✂️ Splitter"]
        SPLIT --> EMBED["🔢 Embeddings"]
        EMBED --> DB[("🗄️ ChromaDB")]
    end

    subgraph Chain["RAG-Chain (LCEL)"]
        Q["❓ Frage"] --> RET["🔍 Retriever"]
        DB --> RET
        RET --> FMT["📝 format_docs()"]
        Q --> PROMPT["📋 RAG-Prompt"]
        FMT --> PROMPT
        PROMPT --> LLM["🤖 LLM"]
        LLM --> PARSE["✂️ StrOutputParser"]
        PARSE --> ANS["💬 Antwort"]
    end
'''

mermaid(diagram, width=900)

# 2 | Retriever erstellen
---

Ein **Retriever** ist die Schnittstelle zwischen der Vektordatenbank und der Chain.  
Er empfängt eine Frage und gibt die relevantesten Dokumente zurück.

## Wissensbasis aufbauen

Wir nutzen Agenten-Kurs-Texte als Wissensbasis:

In [4]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# LLM und Embeddings initialisieren
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("LLM und Embeddings initialisiert")

LLM und Embeddings initialisiert


In [5]:
# Wissensdatenbank aufbauen
texte = [
    ("LangChain 1.0 bietet init_chat_model() fuer einheitliche Modell-Initialisierung. "
     "LCEL-Chains verwenden den Pipe-Operator | zur Verkettung von Komponenten. "
     "create_agent() ist die moderne API fuer Agenten in LangChain 1.0+.",
     {"quelle": "langchain_basics", "kategorie": "frameworks"}),
    ("LangGraph ist ein Framework fuer zustandsbasierte Agenten-Workflows. "
     "StateGraph definiert Knoten (Funktionen) und Kanten (Uebergaenge). "
     "Checkpointing speichert den Agenten-Zustand zwischen Aufrufen.",
     {"quelle": "langgraph_basics", "kategorie": "frameworks"}),
    ("RAG steht fuer Retrieval-Augmented Generation. "
     "Dokumente werden in Vektoren umgewandelt und in ChromaDB gespeichert. "
     "Bei einer Anfrage werden die aehnlichsten Dokumente abgerufen und dem LLM uebergeben.",
     {"quelle": "rag_konzepte", "kategorie": "rag"}),
    ("ChromaDB ist eine lokale Vektordatenbank fuer KI-Anwendungen. "
     "Sie unterstuetzt persistente Speicherung und Metadaten-Filter. "
     "Die Integration mit LangChain erfolgt ueber die Chroma-Klasse.",
     {"quelle": "chromadb_info", "kategorie": "rag"}),
    ("LangSmith ist das Monitoring- und Tracing-Tool fuer LangChain-Agenten. "
     "Traces zeigen den vollstaendigen Ausfuehrungspfad eines Agenten. "
     "Evaluierungen helfen bei der systematischen Qualitaetsmessung.",
     {"quelle": "langsmith_info", "kategorie": "monitoring"}),
]

dokumente = [Document(page_content=t, metadata=m) for t, m in texte]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(dokumente)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="kurs_wissensbasis",
    persist_directory="chroma_m10"
)

print(f"Wissensbasis aufgebaut: {vectorstore._collection.count()} Chunks indexiert")

Wissensbasis aufgebaut: 5 Chunks indexiert


# 3 | Similarity Search
---



Bevor wir die vollständige RAG-Chain bauen, testen wir den Retriever direkt.

<p><font color='black' size="5">
Retriever-Konfiguration
</font></p>

| Parameter | Beschreibung | Wert |
|-----------|-------------|------|
| `k` | Anzahl zurückgegebener Dokumente | 3–5 |
| `search_type` | Suchalgorithmus | `"similarity"` (Standard) |
| `score_threshold` | Mindest-Ähnlichkeit (0–1) | Optional, z.B. 0.5 |

In [6]:
# Retriever aus dem Vectorstore erstellen
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Testanfrage
test_frage = "Was ist LangGraph und wie funktioniert es?"
gefundene_docs = retriever.invoke(test_frage)

print(f"Frage: {test_frage}\n")
print(f"Gefundene Dokumente: {len(gefundene_docs)}\n")
for i, doc in enumerate(gefundene_docs):
    print(f"Dokument {i+1}:")
    print(f"  Quelle:  {doc.metadata.get('quelle', '?')}")
    print(f"  Inhalt:  {doc.page_content[:150]}...")
    print()

Frage: Was ist LangGraph und wie funktioniert es?

Gefundene Dokumente: 3

Dokument 1:
  Quelle:  langgraph_basics
  Inhalt:  LangGraph ist ein Framework fuer zustandsbasierte Agenten-Workflows. StateGraph definiert Knoten (Funktionen) und Kanten (Uebergaenge). Checkpointing ...

Dokument 2:
  Quelle:  langsmith_info
  Inhalt:  LangSmith ist das Monitoring- und Tracing-Tool fuer LangChain-Agenten. Traces zeigen den vollstaendigen Ausfuehrungspfad eines Agenten. Evaluierungen ...

Dokument 3:
  Quelle:  chromadb_info
  Inhalt:  ChromaDB ist eine lokale Vektordatenbank fuer KI-Anwendungen. Sie unterstuetzt persistente Speicherung und Metadaten-Filter. Die Integration mit LangC...



In [7]:
# Similarity Search mit Score (fuer Debugging)
ergebnisse = vectorstore.similarity_search_with_score(
    "Wie funktioniert ChromaDB?",
    k=3
)

mprint("## Similarity Search mit Scores\n")
rows = ["| Nr | Quelle | Score | Inhalt |", "|-----|--------|-------|--------|"]
for i, (doc, score) in enumerate(ergebnisse):
    quelle = doc.metadata.get("quelle", "?")
    inhalt = doc.page_content[:60].replace("\n", " ")
    rows.append(f"| {i+1} | {quelle} | {score:.4f} | {inhalt}... |")
mprint("\n".join(rows))

## Similarity Search mit Scores


| Nr | Quelle | Score | Inhalt |
|-----|--------|-------|--------|
| 1 | chromadb_info | 0.4346 | ChromaDB ist eine lokale Vektordatenbank fuer KI-Anwendungen... |
| 2 | rag_konzepte | 0.9870 | RAG steht fuer Retrieval-Augmented Generation. Dokumente wer... |
| 3 | langchain_basics | 1.3767 | LangChain 1.0 bietet init_chat_model() fuer einheitliche Mod... |

# 4 | RAG-Chain bauen (LCEL)
---



Jetzt bauen wir die vollständige RAG-Chain mit **LCEL** (LangChain Expression Language).



<p><font color='black' size="5">
Chain-Architektur
</font></p>


```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
```

**Erklärung:**
1. `retriever | format_docs` – Holt Dokumente und formatiert sie als Text
2. `RunnablePassthrough()` – Leitet die Frage unverändert weiter
3. `rag_prompt` – Baut den Prompt mit Kontext und Frage
4. `llm` – Sendet an GPT-4o-mini
5. `StrOutputParser()` – Extrahiert den Text aus der Antwort

In [8]:
# RAG-Prompt aus Prompt-Template laden
rag_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m13_rag_prompt.md", mode="T")

# Hilfsfunktion: Dokumente formatieren
def format_docs(docs):
    return (chr(10) * 2).join(doc.page_content for doc in docs)

# RAG-Chain zusammenbauen (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG-Chain bereit")


HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/ralf-42/Agenten/main/05_prompt/m13_rag_prompt.md

In [ ]:
# RAG-Chain testen
test_fragen = [
    "Was ist LangChain und welche Modell-Initialisierung wird empfohlen?",
    "Wie unterscheidet sich LangGraph von LangChain?",
    "Welche Vektordatenbank eignet sich fuer lokale RAG-Systeme?",
]

for frage in test_fragen:
    print(f"Frage: {frage}")
    antwort = rag_chain.invoke(frage)
    print(f"Antwort: {antwort}")
    print("-" * 60)
    print()

# 5 | RAG im LangSmith
---



LangSmith zeigt den vollständigen Trace der RAG-Chain – ideal zum Debuggen, wenn Antworten nicht passen.

**Was LangSmith anzeigt**

| Ebene | Inhalt | Nützlich für |
|-------|--------|-------------|
| **Chain-Ebene** | Input/Output der gesamten Chain | Gesamtbild |
| **Retriever-Ebene** | Welche Dokumente gefunden wurden | Retrieval-Qualität |
| **Prompt-Ebene** | Exakter Prompt (mit Kontext) | Prompt-Debugging |
| **LLM-Ebene** | Tokens, Kosten, Latenz | Performance |

**Trace mit run_name**

```python
antwort = rag_chain.with_config(run_name="RAG-Kursassistent").invoke(frage)
```

In [ ]:
# RAG mit LangSmith-Tracing und run_name
run_cfg = {"run_name": "RAG-Kursassistent", "tags": ["rag", "m13"]}

frage = "Erklaere mir, wie LCEL-Chains in LangChain funktionieren."

antwort = rag_chain.with_config(**run_cfg).invoke(frage)

# Gefundene Dokumente separat anzeigen
kontext_docs = retriever.invoke(frage)

mprint(f"""
## RAG-Ergebnis mit LangSmith-Trace

**Frage:** {frage}

**Verwendete Quellen ({len(kontext_docs)} Dokumente):**
""")

for i, doc in enumerate(kontext_docs):
    print(f"  {i+1}. {doc.metadata.get('quelle', '?')} – {doc.page_content[:80]}...")

mprint(f"""

**Antwort:**

{antwort}

> Trace verfuegbar in LangSmith unter Projekt: M13-RAG-Chain
""")

# 6 | RAG-Optimierung
---



*Häufige Probleme und Lösungen*

| Problem | Symptom | Lösung |
|---------|---------|--------|
| Falsche Dokumente | Irrelevante Antworten | Chunk-Größe reduzieren, k erhöhen |
| Halluzinationen | Fakten nicht in Quellen | Temperatur auf 0.0, strengere Prompts |
| Langsame Antworten | Hohe Latenz | Weniger Chunks (k), kleinere Modelle |
| Zu viel Kontext | Token-Limit | Komprimierung, k reduzieren |
| Inkonsistente Antworten | Unterschiedliche Resultate | temperature=0.0 setzen |

**Optimierungsstrategie**

```
1. Mit kleinem k starten (k=3)
2. LangSmith-Traces analysieren
3. Retrieval-Qualität prüfen (werden relevante Docs gefunden?)
4. Prompt anpassen (präziser, Quellenbindung erzwingen)
5. chunk_size experimentieren (200–500)
```

In [ ]:
# Optimierter Retriever mit Score-Threshold
retriever_opt = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

# Optimierte RAG-Chain
rag_chain_opt = (
    {"context": retriever_opt | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Vergleich: Standard vs. Optimiert
frage_test = "Was weisst du ueber Vektordatenbanken?"

docs_std = retriever.invoke(frage_test)
docs_opt = retriever_opt.invoke(frage_test)

print(f"Standard Retriever (k=3):    {len(docs_std)} Dokumente")
print(f"Optimierter Retriever (k=4): {len(docs_opt)} Dokumente")

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M13-RAG-Chain", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
Biografien-RAG-System
</font></p>

Bauen Sie ein RAG-System für die Biografien-Dateien aus dem Kurs-Datensatz.

**Teilaufgaben:**
1. Laden Sie die Biografien-Dateien aus `../02_daten/01_text/` (biografien_1.txt, biografien_2.md)
2. Erstellen Sie eine persistente ChromaDB-Collection "biografien"
3. Bauen Sie eine RAG-Chain mit dem Standard-RAG-Prompt (`load_prompt("../05_prompt/m10_rag_prompt.md")`)
4. Beantworten Sie 3 Fragen zu den Biografien und zeigen Sie die Quellen an

**Bonus:** Fügen Sie LangSmith-Tracing mit `run_name` und `tags` hinzu.
Analysieren Sie in LangSmith, welche Dokumente bei welcher Frage gefunden wurden.